# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook walks you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (as an object, not a dict)
name = dataset.metadata.name
description = dataset.metadata.description
print(f"{name}: {description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all record sets and their fields, referencing by `@id` as required.

In [ ]:
# List all RecordSets and their Fields by @id
record_sets = dataset.metadata.recordSets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    fields = rs['fields'] if 'fields' in rs else []
    for fld in fields:
        print(f"  Field: {fld['@id']} (name: {fld.get('name', 'N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

All references use `@id`. We'll extract all record sets, then display a sample from the main record set.

In [ ]:
# List all record sets' @id
record_set_ids = [rs['@id'] for rs in record_sets]
print('Record Set @ids:', record_set_ids)

# Try to load records from all record sets
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for {rs_id}")

# Select the primary record set for preview (first)
main_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_rs_id:
    print(f"Columns for {main_rs_id}:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record set found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on specific criteria
- Normalizing numeric fields
- Grouping or categorizing

For demonstration, let's select a numeric clinical field using its `@id`.

In [ ]:
# Assume we have a numeric field, for example 'cr:age' (replace with correct @id from the record set/field listing above)
df = dataframes[main_rs_id] if main_rs_id else pd.DataFrame()

# Identify a numeric field @id
numeric_field_id = None
for rs in record_sets:
    if rs['@id'] == main_rs_id:
        for fld in rs.get('fields', []):
            if fld.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_field_id = fld['@id']
                break
        break

if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    # Filtering: Choose a threshold for demonstration
    threshold = 40  # e.g., filter Age > 40
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Group by another field
        group_field_id = None
        for fld in rs.get('fields', []):
            # Pick a categorical field
            if fld.get('dataType') == 'schema:Text':
                group_field_id = fld['@id']
                break
        
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print(f"Numeric field {numeric_field_id} not present in dataframe columns.")

## 5. Visualization
Visualize distribution of a numeric field and its relation to a categorical attribute.

We'll use matplotlib for basic plotting.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot if group_field_id exists
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        plt.scatter(df[group_field_id], df[numeric_field_id], alpha=0.7)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- This notebook utilized the `mlcroissant` library to load metadata and records from the FAIR^2 dataset schema, referencing all entities by their `@id`.
- We explored available record sets and fields, loaded data, performed basic EDA including filtering, normalization, and grouping, then visualized key attributes.
- The dataset focuses on second primary colorectal cancer in survivors, with rich clinical, anatomical, and molecular detail for further exploration.

For more robust analysis or modeling, continue with additional statistical exploration or implement domain-specific pipelines.